# <font color="#ff3838ff" face="Palatino Linotype">**PIPELINE RISET: PUPIL DATASET PROCESSOR (PDP)**</font>

<details>
<summary>📌 <b>Lihat Diagram Pipeline Riset SOP (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacaploki-pixel/pupil/blob/main/assets/img/pipeline_riset.webp?raw=true" width="100%" alt="Pipeline Riset SOP">
</p>
</details>


## <font color="#38b6ff" face="Palatino Linotype">**FASE 1: DATA PREPARATION & CLEANING**</font>

<details>
<summary>📌 <b>Lihat Infografis FASE 1: Data Preparation & Cleaning (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacaploki-pixel/pupil/blob/main/assets/img/fase1.webp?raw=true" width="100%" alt="FASE 1 Data Preparation & Cleaning">
</p>
</details>

Tahap penyiapan data awal dari video mentah original hingga menjadi kepingan gambar *PNG Lossless* yang terisolasi pada jendela Segmen Optimal (30 Detik), serta pra-pemrosesan pemotongan area mata (ROI) yang stabil.

**Cakupan Standar Operasional Prosedur (SOP):**
* **SOP-01 (Inisialisasi Environment & Konfigurasi Workspace):** Penyiapan sesi komputasi dan pemetaan dataset Google Drive.
* **SOP-02 (Ekstraksi Direct Frame PNG Lossless):** Pencarian rentang Segmen Optimal (30 Detik) (kedipan minimal) dan ekstraksi *direct frame PNG* tanpa kompresi.
* **SOP-03 (Pra-Pemrosesan Grayscale & Stabilized Crop ROI):** Pemotongan area mata dengan fitur *EMA Stabilization* bebas shaking.


In [ ]:
# @title 🛠️ SOP-01: Inisialisasi Environment & Konfigurasi Workspace
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Hubungkan ke Google Drive
drive.mount('/content/drive')

# 2. Folder Utama tempat HASIL output akan disimpan
ROOT_DIR = '/content/drive/MyDrive/Hibah Penelitian/Outputs'
os.makedirs(ROOT_DIR, exist_ok=True)
print(f"[INFO] Folder penyimpanan utama disiapkan di: {ROOT_DIR}\n")

# 3. Memindai Folder Hibah Penelitian secara Presisi
HIBAH_ROOT = '/content/drive/MyDrive/Hibah Penelitian'
if not os.path.exists(HIBAH_ROOT):
    matches = glob.glob('/content/drive/MyDrive/*[Hh][Ii][Bb][Aa][Hh]*')
    if matches:
        HIBAH_ROOT = matches[0]

folder_map = {}
if os.path.exists(HIBAH_ROOT):
    for root, dirs, files in os.walk(HIBAH_ROOT):
        if 'Hibah Penelitian/Outputs' in root:
            continue
        video_in_dir = [f for f in files if f.lower().endswith(('.mp4', '.avi', '.mkv'))]
        if video_in_dir:
            rel_path = os.path.relpath(root, HIBAH_ROOT)
            label = rel_path if rel_path != '.' else 'Folder Utama Hibah'
            folder_map[label] = root

if not folder_map:
    print("[WARN] Tidak ditemukan folder video di dalam Hibah Penelitian.")
    print("[INFO] Menggunakan path default...")
    global DEBUGGING_MODE
    DEBUGGING_MODE = 'Yes'
    vid_path = '/content/drive/MyDrive/Hibah Penelitian/original/10-06-2026/001.MP4'
    vid_name = os.path.splitext(os.path.basename(vid_path))[0]
    RES_DIR = os.path.join(ROOT_DIR, vid_name)
    dir_raw = os.path.join(RES_DIR, '1_Raw_Frames_SOP01&02')
    dir_gray = os.path.join(RES_DIR, '2_Grayscale_ROI_SOP03')
    dir_mask = os.path.join(RES_DIR, '3_Mask_Pupil_SOP04')
    dir_hasil = os.path.join(RES_DIR, '4_Hasil_Analisis_SOP05_06')
    for d in [dir_raw, dir_gray, dir_mask, dir_hasil]:
        os.makedirs(d, exist_ok=True)
else:
    folder_options = [(lbl, path) for lbl, path in sorted(folder_map.items())]
    
    folder_dropdown = widgets.Dropdown(
        options=folder_options,
        description='1. Tanggal Sesi:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='70%')
    )
    
    video_dropdown = widgets.Dropdown(
        options=[],
        description='2. Responden:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='70%')
    )
    

    debugging_dropdown = widgets.Dropdown(
        options=['Yes', 'No'],
        value='Yes',
        description='3. Debug Mode:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='70%')
    )
    
    def update_debugging_mode(change):
        global DEBUGGING_MODE
        DEBUGGING_MODE = debugging_dropdown.value
        
    debugging_dropdown.observe(update_debugging_mode, names='value')
    
    output_box = widgets.Output()
    
    def update_video_options(change):
        selected_folder = folder_dropdown.value
        vids = []
        if selected_folder and os.path.exists(selected_folder):
            for f in sorted(os.listdir(selected_folder)):
                if f.lower().endswith(('.mp4', '.avi', '.mkv')):
                    vids.append((f, os.path.join(selected_folder, f)))
        video_dropdown.options = vids
        if vids:
            video_dropdown.value = vids[0][1]
            
    def update_selected_video(change):
        global vid_path, vid_name, RES_DIR, dir_raw, dir_gray, dir_mask, dir_hasil
        selected_vid = video_dropdown.value
        if selected_vid:
            vid_path = selected_vid
            vid_name = os.path.splitext(os.path.basename(vid_path))[0]
            RES_DIR = os.path.join(ROOT_DIR, vid_name)
            dir_raw = os.path.join(RES_DIR, '1_Raw_Frames_SOP01&02')
            dir_gray = os.path.join(RES_DIR, '2_Grayscale_ROI_SOP03')
            dir_mask = os.path.join(RES_DIR, '3_Mask_Pupil_SOP04')
            dir_hasil = os.path.join(RES_DIR, '4_Hasil_Analisis_SOP05_06')
            
            for d in [dir_raw, dir_gray, dir_mask, dir_hasil]:
                os.makedirs(d, exist_ok=True)
                
            with output_box:
                clear_output()
                print(f"======================================================================")
                print(f"[RESPONDEN AKTIF: {vid_name}]")
                print(f"======================================================================")
                print(f"🎯 Video Terpilih : {vid_path}")
                print(f"📁 Folder Output  : {RES_DIR}")
                print(f"🔗 Google Drive   : https://drive.google.com/drive/my-drive")
                print(f"✅ Workspace siap diproses pada SOP-01 & SOP-02!\n")
                
    folder_dropdown.observe(update_video_options, names='value')
    video_dropdown.observe(update_selected_video, names='value')
    
    print("📌 PILIH DATASET TERFOKUS (HIBAH PENELITIAN):\n")
    display(folder_dropdown)
    display(video_dropdown)
    display(debugging_dropdown)
    display(output_box)
    
    update_video_options(None)
    update_selected_video(None)
    update_debugging_mode(None)



In [ ]:
# @title ✂️ SOP-02: Ekstraksi Otomatis 30 Detik Segmen Stabil
def check_blink_fast(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(blurred)
    cx, cy = min_loc
    
    roi_size = 800
    h, w = gray.shape
    x_min, y_min = max(0, cx - roi_size // 2), max(0, cy - roi_size // 2)
    x_max, y_max = min(w, cx + roi_size // 2), min(h, cy + roi_size // 2)
    roi_blurred = blurred[y_min:y_max, x_min:x_max]
    
    thresh_val = min(min_val + 28, 130)
    _, thresh = cv2.threshold(roi_blurred, thresh_val, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    radius = 0
    if contours:
        valid_contours = [c for c in contours if cv2.contourArea(c) > 100]
        if valid_contours:
            best_contour = max(valid_contours, key=cv2.contourArea)
            _, radius = cv2.minEnclosingCircle(best_contour)
    return radius * 2

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-01 & SOP-02: EKSTRAKSI 30 DETIK EMAS")
print(f"======================================================================")
print(f"[STEP 1/2] Memindai kedipan minimal pada video: {vid_path}")

cap = cv2.VideoCapture(vid_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
target_frames = 1500

if globals().get('DEBUGGING_MODE', 'No') == 'Yes':
    target_frames = 300
    print("[DEBUG MODE AKTIF] Target dibatasi menjadi 300 frame!")

if total_frames <= target_frames:
    start_f, end_f = 0, total_frames
else:
    blinks = []
    frame_idx = 0
    with tqdm(total=total_frames//5, desc=f"Scanning [{vid_name}]") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret: break
            if frame_idx % 5 == 0:
                if check_blink_fast(frame) < 5:
                    blinks.append(frame_idx)
                pbar.update(1)
            frame_idx += 1
            
    best_start = 0
    min_blinks = float('inf')
    for start_i in range(0, total_frames - target_frames, 50):
        end_i = start_i + target_frames
        blink_count = sum(1 for b in blinks if start_i <= b <= end_i)
        if blink_count < min_blinks:
            min_blinks = blink_count
            best_start = start_i
            
    start_f = best_start
    end_f = best_start + target_frames

print(f"\n[STEP 2/2] Rentang Segmen Optimal (30 Detik) ditemukan pada Frame {start_f} sampai {end_f}.")
print(f"Mengekstrak 1.500 Frame PNG Lossless langsung ke folder: {dir_raw}")

cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
frames_to_read = end_f - start_f

for i in tqdm(range(frames_to_read), desc=f"Extracting [{vid_name}]"):
    ret, frame = cap.read()
    if not ret: break
    raw_name = f"{vid_name}_frame_{i:04d}.png"
    cv2.imwrite(os.path.join(dir_raw, raw_name), frame)

cap.release()
print(f"\n======================================================================")
print(f"✅ DATASET SOP-01 & SOP-02 BERHASIL TERSIMPAN DI GOOGLE DRIVE!")
print(f"📁 Nama Folder Drive : 1_Raw_Frames_SOP01&02")
print(f"📍 Drive Local Path  : {dir_raw}")
print(f"🔗 Akses Google Drive: https://drive.google.com/drive/my-drive")
print(f"======================================================================")


In [ ]:
# @title 🎬 SOP-03: Pra-Pemrosesan Grayscale & Stabilized Crop ROI 800x800
print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-03: PRA-PEMROSESAN (GRAYSCALE & STABILIZED CROP ROI)")
print(f"======================================================================")

raw_files = sorted(os.listdir(dir_raw))
print(f"[INFO] Memproses {len(raw_files)} frame ke 2_Grayscale_ROI_SOP03 (Stabilized EMA)...")

roi_size = 800
cx_smooth, cy_smooth = None, None
alpha = 0.05  # EMA smoothing factor untuk kestabilan ROI tanpa shaking

for filename in tqdm(raw_files, desc=f"Grayscale & ROI [{vid_name}]"):
    frame = cv2.imread(os.path.join(dir_raw, filename))
    if frame is None: continue
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    blur = cv2.GaussianBlur(gray, (15, 15), 0)
    _, thresh = cv2.threshold(blur, 40, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        c = max(contours, key=cv2.contourArea)
        (x_c, y_c), _ = cv2.minEnclosingCircle(c)
        if cx_smooth is None or cy_smooth is None:
            cx_smooth, cy_smooth = x_c, y_c
        else:
            cx_smooth = (alpha * x_c) + ((1 - alpha) * cx_smooth)
            cy_smooth = (alpha * y_c) + ((1 - alpha) * cy_smooth)
    else:
        if cx_smooth is None:
            cx_smooth, cy_smooth = gray.shape[1]/2, gray.shape[0]/2
            
    # Calculate bounding box
    x1 = int(cx_smooth - roi_size/2)
    y1 = int(cy_smooth - roi_size/2)
    x2 = x1 + roi_size
    y2 = y1 + roi_size
    
    # Boundary padding if exceeding frame
    pad_top, pad_bottom, pad_left, pad_right = 0, 0, 0, 0
    
    if y1 < 0:
        pad_top = -y1
        y1 = 0
    if y2 > gray.shape[0]:
        pad_bottom = y2 - gray.shape[0]
        y2 = gray.shape[0]
        
    if x1 < 0:
        pad_left = -x1
        x1 = 0
    if x2 > gray.shape[1]:
        pad_right = x2 - gray.shape[1]
        x2 = gray.shape[1]
        
    roi = gray[y1:y2, x1:x2]
    
    if pad_top > 0 or pad_bottom > 0 or pad_left > 0 or pad_right > 0:
        roi = cv2.copyMakeBorder(roi, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REPLICATE)
        
    cv2.imwrite(os.path.join(dir_gray, filename), roi)

print("[INFO] Tahap SOP-03 Selesai.")

## <font color="#38b6ff" face="Palatino Linotype">**FASE 2: PENGEMBANGAN & DEBUGGING PIPELINE OTOMASI**</font>

<details>
<summary>📌 <b>Lihat Infografis FASE 2: Pengembangan & Debugging Pipeline Otomasi (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacaploki-pixel/pupil/blob/main/assets/img/fase2.webp?raw=true" width="100%" alt="FASE 2 Pengembangan & Debugging Pipeline Otomasi">
</p>
</details>

Tahap segmentasi pupil, ekstraksi fitur osilasi hippus temporal, dan analisis medis tingkat lanjut.

**Cakupan Standar Operasional Prosedur (SOP):**
* **SOP-04 (Segmentasi Mask Pupil & Video Verifikasi):** Segmentasi biner pupil, *Anatomical Reconstruction*, dan rendering 3 video MP4 real-time.
* **SOP-05 (Tracking Pupil & Analisis Sinyal):** Ekstraksi fitur medis hippus, ekspor berkas CSV, dan rendering grafik PNG.
* **SOP-06 (Generator Laporan PDF Klinis):** Penyusunan dan penertiban dokumen cetak PDF Diagnostik Klinis Responden.


In [ ]:
# @title 🛠️ SOP-04 (Langkah 1): Definisi Core Engine OpenCV
# ======================================================================
# SOP-04 (Langkah 1): DETEKSI PUPIL (VERIFIED PRECISION & BLINK RESISTANT)
# ======================================================================
import cv2, os, math
import numpy as np

def detect_pupil_opencv_sop05(roi_gray, px=None, py=None, last_valid_pupil=None, frames_lost=0):
    heavy_blur = cv2.medianBlur(roi_gray, 25)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(heavy_blur)
    if px is None or py is None:
        px, py = min_loc

    # Eyeball closure / blink check (min_val > 175)
    if min_val > 175:
        return np.zeros_like(roi_gray), 0.0, 0, 0, 0.0, None, None

    fine_blur = cv2.GaussianBlur(roi_gray, (7, 7), 0)
    pupil_thresh = min_val + 25  # Validated optimal threshold (+25)
    _, thresh = cv2.threshold(fine_blur, pupil_thresh, 255, cv2.THRESH_BINARY_INV)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh_m = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    thresh_m = cv2.morphologyEx(thresh_m, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(thresh_m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    mask = np.zeros_like(roi_gray)
    diam = 0.0
    cx, cy = 0, 0
    angle_deg = 0.0
    current_valid_pupil = last_valid_pupil
    best_ellipse = None

    # Reset tracking anchor post-blink to prevent lag/flicker
    if frames_lost > 4:
        last_valid_pupil = None

    max_dist = 180 if last_valid_pupil is not None else 350

    if contours:
        valid_contours = []
        for c in contours:
            area = cv2.contourArea(c)
            if 2000 < area < 30000:
                (cx_c, cy_c), radius = cv2.minEnclosingCircle(c)
                if cy_c < 130:
                    continue
                    
                if last_valid_pupil is not None:
                    dist_anchor = np.hypot(cx_c - last_valid_pupil[0], cy_c - last_valid_pupil[1])
                else:
                    dist_anchor = np.hypot(cx_c - px, cy_c - py)

                perimeter = cv2.arcLength(c, True)
                circ = 4 * np.pi * (area / (perimeter * perimeter)) if perimeter > 0 else 0
                
                if dist_anchor < max_dist and circ > 0.25:
                    valid_contours.append((c, area, dist_anchor, circ))

        if valid_contours:
            best_c = min(valid_contours, key=lambda item: item[2] - 50 * item[3])[0]
            
            area = cv2.contourArea(best_c)
            diam = 2.0 * np.sqrt(area / np.pi)
            
            if len(best_c) >= 5:
                best_ellipse = cv2.fitEllipse(best_c)
                cv2.ellipse(mask, best_ellipse, 255, thickness=cv2.FILLED)
                (x, y), (a, b), angle_deg = best_ellipse
                cx, cy = int(x), int(y)
            else:
                (x, y), radius = cv2.minEnclosingCircle(best_c)
                cv2.circle(mask, (int(x), int(y)), int(radius), 255, thickness=cv2.FILLED)
                cx, cy = int(x), int(y)
                best_ellipse = ((x, y), (radius*2, radius*2), 0.0)

            if diam > 220 or diam < 40:
                diam = 0
                cx, cy = 0, 0
                mask[:] = 0
                best_ellipse = None
            else:
                current_valid_pupil = (cx, cy, diam)

    return mask, diam, cx, cy, angle_deg, current_valid_pupil, best_ellipse
print("[INFO] Fungsi detect_pupil_opencv_sop05 berhasil didefinisikan dengan perbaikan presisi.")


In [ ]:
# @title ⚙️ SOP-04 (Langkah 2): Eksekusi Segmentasi Masker Biner PNG (Efisiensi Storage)
# ======================================================================
# SOP-04 (Langkah 2): Eksekusi Segmentasi Biner (Hanya 1_Mask_Biner)
# ======================================================================
import os, cv2
import numpy as np
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-04 (LANGKAH 2): SEGMENTASI MASK PUPIL")
print(f"======================================================================")

sub_mask = os.path.join(dir_mask, "1_Mask_Biner")
os.makedirs(sub_mask, exist_ok=True)

diameters = []
centers = []
angles = []
ellipses = []
masks_in_memory = []
gray_frames_in_memory = []
gray_files = sorted(os.listdir(dir_gray))

last_valid_pupil = None
frames_lost = 0

print(f"[INFO] Mengolah {len(gray_files)} frame (Simpan 1_Mask_Biner ke Google Drive)...")

for i, filename in enumerate(tqdm(gray_files, desc=f"Segmentasi [{vid_name}]")):
    roi_gray = cv2.imread(os.path.join(dir_gray, filename), cv2.IMREAD_GRAYSCALE)
    if roi_gray is None:
        diameters.append(0)
        centers.append((0, 0))
        angles.append(0)
        ellipses.append(None)
        masks_in_memory.append(np.zeros((800, 800), dtype=np.uint8))
        gray_frames_in_memory.append(np.zeros((800, 800), dtype=np.uint8))
        frames_lost += 1
        continue

    mask, diam, cx, cy, angle_deg, last_valid_pupil, ellipse = detect_pupil_opencv_sop05(
        roi_gray=roi_gray, px=None, py=None, last_valid_pupil=last_valid_pupil, frames_lost=frames_lost
    )

    if diam > 0:
        frames_lost = 0
    else:
        frames_lost += 1

    diameters.append(diam)
    centers.append((cx, cy))
    angles.append(round(angle_deg, 2))
    ellipses.append(ellipse)
    masks_in_memory.append(mask)
    gray_frames_in_memory.append(roi_gray)

    cv2.imwrite(os.path.join(sub_mask, filename), mask)

print("[INFO] Segmentasi dan penulisan Masker Biner selesai (Efisiensi Storage Aktif).")


In [ ]:
# @title 🎥 SOP-04 (Langkah 3): Render Berkas Video Verifikasi MP4 (1 Video Terpadu Utama)
# ======================================================================
# SOP-04 (Langkah 3): RENDERING 1 VIDEO TERPADU UTAMA
# ======================================================================
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

print(f"\n[INFO] Memulai perenderan berkas 1 Video Verifikasi Terpadu MP4...")

gray_files = sorted(os.listdir(dir_gray))

# === OUTLIER REJECTION & TREND ===
raw_d = np.array(diameters, dtype=np.float64)
raw_d[raw_d == 0] = np.nan

temp = np.copy(raw_d)
mask_nan = np.isnan(temp)
if np.any(~mask_nan):
    temp[mask_nan] = np.interp(np.flatnonzero(mask_nan), np.flatnonzero(~mask_nan), temp[~mask_nan])
else:
    temp[:] = 92.0

fps_asumsi = 30.0
window = 150
pad_w = window // 2
padded = np.pad(temp, (pad_w, pad_w), mode='edge')
trend = np.zeros_like(temp)
for i in range(len(temp)):
    trend[i] = np.median(padded[i:i+window])
    
abs_diff = np.abs(raw_d - trend) / (trend + 1e-5)
outliers = (abs_diff > 0.15) | np.isnan(raw_d)

kernel = np.ones(11, dtype=bool)
dilated_outliers = np.convolve(outliers, kernel, mode='same') > 0

clean_d = np.copy(raw_d)
clean_d[dilated_outliers] = np.nan

s_diams = pd.Series(clean_d).interpolate(limit_direction='both')
if s_diams.isna().all(): s_diams = pd.Series([92.0]*len(clean_d))
s_diams = s_diams.bfill().ffill()

s_diams_smooth = s_diams.rolling(window=5, min_periods=1, center=True).mean()
s_trend_smooth = pd.Series(trend).rolling(window=15, min_periods=1, center=True).mean()

d_array = s_diams_smooth.to_numpy()
peaks, _ = find_peaks(d_array, distance=15, prominence=0.5)

baseline = np.mean(d_array)
max_d = np.max(d_array)
min_d = np.min(d_array)
amp_pct = ((max_d - min_d) / baseline) * 100 if baseline > 0 else 0
freq = (len(peaks) / len(d_array)) * fps_asumsi

# === RENDERING 1 VIDEO TERPADU (Sama seperti lokal) ===
path_v1 = os.path.join(dir_mask, f"{vid_name}_Video_Tracking_dan_Grafik_Realtime.mp4")

FRAME_W, FRAME_H = 800, 800
GRAPH_W, GRAPH_H = 1600, 600
CANVAS_W, CANVAS_H = 1600, 1400
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer_v1 = cv2.VideoWriter(path_v1, fourcc, fps_asumsi, (CANVAS_W, CANVAS_H))

g_max = max(np.max(d_array), np.max(trend)) + 5
g_min = min(np.min(d_array), np.min(trend)) - 5

for i in tqdm(range(len(gray_files)), desc="Rendering Video Terpadu Utama"):
    roi_gray = gray_frames_in_memory[i]
    mask_frame = masks_in_memory[i]

    bgr_roi = cv2.cvtColor(roi_gray, cv2.COLOR_GRAY2BGR)
    if bgr_roi.shape[0] != FRAME_H or bgr_roi.shape[1] != FRAME_W:
        bgr_roi = cv2.resize(bgr_roi, (FRAME_W, FRAME_H))

    raw_cx, raw_cy = centers[i]
    raw_diam = diameters[i]
    is_blink = dilated_outliers[i]

    if is_blink or raw_diam == 0:
        cv2.putText(bgr_roi, "BLINK", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
    else:
        overlay_radius = int(raw_diam / 2.0)
        cv2.circle(bgr_roi, (raw_cx, raw_cy), overlay_radius, (0, 255, 0), 2, cv2.LINE_AA)
        cv2.circle(bgr_roi, (raw_cx, raw_cy), 3, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.putText(bgr_roi, f"PUPIL D={raw_diam:.1f}px", (max(10, raw_cx-80), max(30, raw_cy-overlay_radius-15)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
    
    cv2.putText(bgr_roi, f"1. Original ROI + Tracking [Frame {i}]", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    mask_bgr = cv2.cvtColor(mask_frame, cv2.COLOR_GRAY2BGR)
    if mask_bgr.shape[0] != FRAME_H or mask_bgr.shape[1] != FRAME_W:
        mask_bgr = cv2.resize(mask_bgr, (FRAME_W, FRAME_H))
    cv2.putText(mask_bgr, "2. Realtime Binary Mask", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    top_row = np.hstack((bgr_roi, mask_bgr))

    fig, ax = plt.subplots(figsize=(16, 6), dpi=100)
    x_vals = np.arange(i + 1)
    ax.plot(x_vals, d_array[:i+1], color='#3176b5', linewidth=2.5, label='Diameter Pupil (px)')
    ax.plot(x_vals, s_trend_smooth.iloc[:i+1], color='#f13c3c', linestyle='--', linewidth=2, label='Trend Baseline')
    
    peaks_up_to_i = [p for p in peaks if p <= i]
    if peaks_up_to_i:
        ax.scatter(peaks_up_to_i, d_array[peaks_up_to_i], color='#e87a20', s=70, zorder=5, label='Puncak Hippus')
        
    ax.set_xlim(0, len(gray_files))
    ax.set_ylim(g_min, g_max)
    ax.set_title(f"Temporal Analysis (Pupillary Hippus) - Responden {vid_name} | Frekuensi: {freq:.2f} Hz | Fluktuasi: {amp_pct:.2f} %", fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel("Frame ID (30 FPS)", fontsize=11)
    ax.set_ylabel("Diameter Pupil (px)", fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=10)
    
    fig.tight_layout()
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    graph_img = cv2.cvtColor(rgba, cv2.COLOR_RGBA2BGR)
    plt.close(fig)

    if graph_img.shape[1] != GRAPH_W or graph_img.shape[0] != GRAPH_H:
        graph_img = cv2.resize(graph_img, (GRAPH_W, GRAPH_H))

    canvas = np.vstack((top_row, graph_img))
    writer_v1.write(canvas)

writer_v1.release()

print("======================================================================")
print(" RENDER 1 VIDEO TERPADU UTAMA SELESAI!")
print("Folder Utama Drive : 3_Mask_Pupil_SOP04")
print("======================================================================")


In [ ]:
# @title 📈 SOP-05: Tracking Pupil & Analisis Sinyal Temporal Fluktuasi Hippus
# ======================================================================
# SOP-05: ANALISIS TEMPORAL HIPPUS & EKSPOR DATA (SINKRON DENGAN SOP-04)
# ======================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-05: ANALISIS TEMPORAL HIPPUS & EKSPOR DATA")
print(f"======================================================================")

cx_list = [c[0] for c in centers]
cy_list = [c[1] for c in centers]
df = pd.DataFrame({
    'Frame': range(len(diameters)),
    'Center_X': cx_list,
    'Center_Y': cy_list,
    'Ellipse_Angle': angles,
    'Diameter_px': diameters
})

# 1. Filter Outlier & Interpolasi (Logika Presisi Identik SOP-04)
fps_asumsi = 30.0 # Default fallback, 50.0 atau 90.0 bisa digunakan
raw_d = df['Diameter_px'].replace(0, np.nan).to_numpy(dtype=np.float64)

temp = np.copy(raw_d)
mask_nan = np.isnan(temp)
if np.any(~mask_nan):
    temp[mask_nan] = np.interp(np.flatnonzero(mask_nan), np.flatnonzero(~mask_nan), temp[~mask_nan])
else:
    temp[:] = 92.0

window = 150
pad_w = window // 2
padded = np.pad(temp, (pad_w, pad_w), mode='edge')
trend = np.zeros_like(temp)
for i in range(len(temp)):
    trend[i] = np.median(padded[i:i+window])
    
abs_diff = np.abs(raw_d - trend) / (trend + 1e-5)
outliers = (abs_diff > 0.15) | np.isnan(raw_d)

kernel = np.ones(11, dtype=bool)
dilated_outliers = np.convolve(outliers, kernel, mode='same') > 0

df['Is_Blink'] = dilated_outliers.astype(int)

clean_d = np.copy(raw_d)
clean_d[dilated_outliers] = np.nan

s_diams = pd.Series(clean_d).interpolate(limit_direction='both')
if s_diams.isna().all(): s_diams = pd.Series([92.0]*len(clean_d))
s_diams = s_diams.bfill().ffill()

# Penapisan Mulus Sesuai SOP-04 (window=5, window=15)
s_diams_smooth = s_diams.rolling(window=5, min_periods=1, center=True).mean()
s_trend_smooth = pd.Series(trend).rolling(window=15, min_periods=1, center=True).mean()

df['Filtered_Diameter_px'] = s_diams_smooth.values
df['Trend_Baseline'] = s_trend_smooth.values

# 2. Ekstraksi Fitur Medis Hippus Identik SOP-04 (distance=15, prominence=0.5)
d_array = s_diams_smooth.to_numpy()
peaks, _ = find_peaks(d_array, distance=15, prominence=0.5)

baseline = np.mean(d_array)
max_d = np.max(d_array)
min_d = np.min(d_array)
amplitude = ((max_d - min_d) / baseline) * 100 if baseline > 0 else 0.0
duration_sec = len(df) / fps_asumsi
frequency = float(len(peaks) / duration_sec) if duration_sec > 0 else 0.0

print(f"\n=== HASIL ANALISIS HIPPUS [{vid_name}] ===")
print(f"- Frekuensi Hippus : {frequency:.2f} Hz (Gelombang/detik)")
print(f"- Fluktuasi Amplitudo: {amplitude:.2f} % (Rentang fluktuasi pupil)")
print(f"- Rata-rata Diameter: {baseline:.2f} px\n")

dir_hasil = os.path.join(ROOT_DIR, f"{vid_name}/4_Hasil_Analisis_SOP05_06")
os.makedirs(dir_hasil, exist_ok=True)

csv_path = os.path.join(dir_hasil, f"{vid_name}_laporan_analisis_sop06.csv")
df.to_csv(csv_path, index=False)

# Render Grafik PNG Identik dengan Visual SOP-04 Video
plt.figure(figsize=(12, 5), dpi=120)
x_vals = df['Frame']
plt.plot(x_vals, df['Filtered_Diameter_px'], color='#3176b5', linewidth=2, label='Diameter Pupil (px)')
plt.plot(x_vals, df['Trend_Baseline'], color='#f13c3c', linestyle='--', linewidth=2, label='Trend Baseline')

if len(peaks) > 0:
    plt.scatter(x_vals.iloc[peaks], df['Filtered_Diameter_px'].iloc[peaks], color='#e87a20', s=60, zorder=5, label='Puncak Hippus')

g_max = max(np.max(d_array), np.max(trend)) + 5
g_min = max(0, min(np.min(d_array), np.min(trend)) - 5)
plt.ylim(g_min, g_max)

plt.title(f'Temporal Analysis (Pupillary Hippus) - Responden {vid_name}\nFrekuensi: {frequency:.2f} Hz | Fluktuasi: {amplitude:.2f} %', fontsize=12, fontweight='bold')
plt.xlabel('Frame Number (30 FPS)')
plt.ylabel('Diameter (px)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()

graph_path = os.path.join(dir_hasil, f"{vid_name}_chart_hippus_sop06.png")
plt.savefig(graph_path)
plt.show()

print(f"\n[INFO] Data CSV dan Grafik Hippus Progresif berhasil disimpan ke Google Drive.")


In [ ]:
# @title 📄 SOP-06: Generator Laporan PDF Diagnostik Klinis Individual
# ======================================================================
# SOP-06: GENERATOR LAPORAN PDF DIAGNOSTIK KLINIS (REPORTLAB FAILSAFE)
# ======================================================================
import os
import pandas as pd
import numpy as np

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-06: GENERATOR LAPORAN PDF DIAGNOSTIK KLINIS")
print(f"======================================================================")

# 1. Pastikan Pustaka ReportLab Tersedia (Failsafe Anti-Crash)
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
except ImportError:
    import subprocess
    print("[INFO] Menginstal pemustakaan reportlab...")
    subprocess.check_call(['pip', 'install', 'reportlab', '--quiet'])
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors

dir_hasil = os.path.join(ROOT_DIR, f"{vid_name}/4_Hasil_Analisis_SOP05_06")
os.makedirs(dir_hasil, exist_ok=True)

pdf_path = os.path.join(dir_hasil, f"{vid_name}_Laporan_Klinis_SOP06.pdf")
graph_path = os.path.join(dir_hasil, f"{vid_name}_chart_hippus_sop06.png")

# 2. Persiapan Data Metrik dari SOP-05
blink_count = int(df['Is_Blink'].sum()) if 'Is_Blink' in df.columns else 0
total_f = len(df)
blink_rate_pct = (blink_count / total_f * 100.0) if total_f > 0 else 0.0

doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
styles = getSampleStyleSheet()

title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=16, leading=20, textColor=colors.HexColor('#003366'))
sub_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=9.5, leading=13, textColor=colors.HexColor('#444444'))
body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontName='Helvetica', fontSize=9, leading=12, textColor=colors.HexColor('#222222'))

story = []
story.append(Paragraph(f"LAPORAN DIAGNOSTIK KLINIS INDIVIDUAL - RESPONDEN {vid_name}", title_style))
story.append(Paragraph(f"<b>Platform:</b> Pupil Dataset Processor (PDP) | <b>Hibah Penelitian:</b> 2026", sub_style))
story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#003366"), spaceAfter=10))

tbl_data = [
    [Paragraph("<b>Parameter Metrik Medis</b>", body_style), Paragraph("<b>Nilai Hasil Analisis</b>", body_style)],
    [Paragraph("Identitas Responden", body_style), Paragraph(f"{vid_name}", body_style)],
    [Paragraph("Total Frame Teranalisis", body_style), Paragraph(f"{total_f} Frame (30 FPS)", body_style)],
    [Paragraph("Rata-rata Diameter Baseline", body_style), Paragraph(f"{baseline:.2f} Piksel (px)", body_style)],
    [Paragraph("Frekuensi Osilasi Hippus", body_style), Paragraph(f"{frequency:.2f} Hz (Gelombang/detik)", body_style)],
    [Paragraph("Fluktuasi Amplitudo Pupil", body_style), Paragraph(f"{amplitude:.2f} %", body_style)],
    [Paragraph("Persentase Kedipan (Blink Rate)", body_style), Paragraph(f"{blink_rate_pct:.2f} % ({blink_count} Frame)", body_style)]
]

t = Table(tbl_data, colWidths=[280, 260])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#EAECEE")),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#BDC3C7")),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE')
]))
story.append(t)
story.append(Spacer(1, 15))

if os.path.exists(graph_path):
    story.append(Paragraph("Grafik Deret-Waktu Sinyal Fluktuasi Pupil (Hippus):", styles['Heading2']))
    story.append(Spacer(1, 5))
    story.append(RLImage(graph_path, width=540, height=225))

doc.build(story)

print(f"======================================================================")
print(f"📄 Nama Berkas PDF: {os.path.basename(pdf_path)}")
print(f"📍 Drive Local Path: {pdf_path}")
print(f"🔗 Akses Google Drive: https://drive.google.com/drive/my-drive")
print(f"======================================================================
")



## <font color="#38b6ff" face="Palatino Linotype">**FASE 3: VALIDASI BENCHMARK LPW & PENYELESAIAN RISET**</font>

<details>
<summary>📌 <b>Lihat Infografis FASE 3: Validasi Benchmark & Evaluasi (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacaploki-pixel/pupil/blob/main/assets/img/fase3.webp?raw=true" width="100%" alt="FASE 3 Infographic">
</p>
</details>

Tahap pengujian mutlak (*cross-validation*) untuk mengukur seberapa akurat algoritma OpenCV kita jika disandingkan dengan label *Ground Truth* publik berstandar internasional (*Labeled Pupils in the Wild* / LPW), serta tahap finalisasi tata kelola dan dokumentasi penelitian.

* **SOP-07 (Validasi Benchmark LPW):** Memutar *dataset* sekunder 90 FPS yang sarat *noise* & gerakan agresif, membandingkannya dengan koordinat *Ground Truth* manual, merender visualisasi komparatif, serta mencetak laporan kuantitatif *Detection Rate* secara absolut.
* **SOP-08 (Penyimpanan & Tata Kelola Dataset):** Pengarsipan seluruh hasil luaran (video, CSV, PDF) ke dalam basis data riset (*Google Drive / Harddisk*) yang berstatus *Restricted / Private*.
* **SOP-09 (Dokumentasi Riset & Publikasi Jurnal):** Penggunaan data luaran SOP 1 hingga 8 sebagai basis data empiris untuk menyusun naskah manuskrip jurnal ilmiah dan laporan akhir riset.


In [ ]:
# @title 🎯 SOP-07 (Langkah 1): Form Pemilihan Dataset LPW
# ======================================================================
# SOP-07 (Langkah 1): PEMILIHAN DATASET LPW (DRIVE AUTOMATIC SCAN)
# ======================================================================
import os
import glob
import ipywidgets as widgets
from IPython.display import display

print("======================================================================")
print(" SOP-07: LPW BENCHMARK VALIDATION (GROUND TRUTH EVALUATION)")
print("======================================================================")

# 1. Deteksi Presisi Jalur LPW Dataset di Google Drive
candidate_paths = [
    "/content/drive/MyDrive/Hibah Penelitian/LPW Dataset",
    os.path.join(HIBAH_ROOT if 'HIBAH_ROOT' in globals() else "/content/drive/MyDrive/Hibah Penelitian", "LPW Dataset"),
    os.path.join(ROOT_DIR if 'ROOT_DIR' in globals() else "/content/drive/MyDrive/Hibah Penelitian/Outputs", "Benchmark_LPW"),
    os.path.join(ROOT_DIR if 'ROOT_DIR' in globals() else "/content/drive/MyDrive/Hibah Penelitian/Outputs", "LPW Dataset")
]

LPW_ROOT = None
for path in candidate_paths:
    if os.path.exists(path):
        subdirs = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)) and not d.startswith("Output")]
        if subdirs:
            LPW_ROOT = path
            break

if not LPW_ROOT:
    LPW_ROOT = "/content/drive/MyDrive/Hibah Penelitian/LPW Dataset"
    os.makedirs(LPW_ROOT, exist_ok=True)
    print(f"[WARN] Folder LPW Dataset disiapkan di: {LPW_ROOT}")
else:
    print(f"[INFO] Berhasil mendeteksi LPW Dataset di: {LPW_ROOT}")

# 2. Deteksi Subfolder Responden LPW (001, 004, 009)
try:
    available_lpw = [d for d in os.listdir(LPW_ROOT) if os.path.isdir(os.path.join(LPW_ROOT, d)) and not d.startswith("Output")]
    available_lpw = sorted(available_lpw)
except FileNotFoundError:
    available_lpw = []

if not available_lpw:
    available_lpw = ["001", "004", "009"]

dropdown_lpw = widgets.Dropdown(
    options=available_lpw,
    value=available_lpw[0],
    description='Target LPW:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

display(dropdown_lpw)
print("\n[INFO] Pilih Responden LPW dari dropdown di atas, lalu jalankan Langkah 2.")


In [ ]:
# @title 📂 SOP-07 (Langkah 2): Parsing Koordinat Ground Truth LPW & Inisialisasi Workspace
# ======================================================================
# SOP-07 (Langkah 2): PARSING GROUND TRUTH LPW (1.txt)
# ======================================================================
import os
import pandas as pd
import numpy as np

lpw_id = dropdown_lpw.value
dir_lpw = os.path.join(LPW_ROOT, str(lpw_id))
gt_file = os.path.join(dir_lpw, "1.txt")
vid_lpw_path = os.path.join(dir_lpw, "1.avi")

print(f"======================================================================")
print(f"[TARGET LPW: {lpw_id}] - MEMUAT GROUND TRUTH & VIDEO")
print(f"======================================================================")

# 1. Parsing GT File
if os.path.exists(gt_file):
    try:
        gt_df = pd.read_csv(gt_file, sep='\s+', header=None, names=['GT_X', 'GT_Y'])
        print(f"[INFO] Berhasil memuat {len(gt_df)} frame dari Ground Truth (1.txt).")
    except Exception as e:
        print(f"[ERROR] Gagal mem-parsing file GT: {e}")
        gt_df = pd.DataFrame(columns=['GT_X', 'GT_Y'])
else:
    print(f"[ERROR] Berkas Ground Truth (1.txt) tidak ditemukan di {dir_lpw}")
    gt_df = pd.DataFrame(columns=['GT_X', 'GT_Y'])

# 2. Verifikasi Video
if os.path.exists(vid_lpw_path):
    print(f"[INFO] Video LPW ditemukan: 1.avi")
else:
    print(f"[ERROR] Video 1.avi tidak ditemukan di {dir_lpw}")

# Output Directory (Format Legacy: Output_LPW_[ID])
out_dir_lpw = os.path.join(dir_lpw, f"Output_LPW_{lpw_id}")
os.makedirs(out_dir_lpw, exist_ok=True)
print(f"[INFO] Folder output siap di: {out_dir_lpw}")


In [ ]:
# @title 🎥 SOP-07 (Langkah 3): Render Video MP4 Grid 4-Panel (2x2 Layout Realtime - Fixed Y-Limits)
# ======================================================================
# SOP-07 (Langkah 3): RENDERING VIDEO MP4 GRID 4-PANEL (2x2 LAYOUT FIXED Y-LIMITS)
# ======================================================================
import cv2, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

print(f"\n[INFO] Mengeksekusi Analisis OpenCV dan Merender Video MP4 Grid 4-Panel (2x2)...")

if 'detect_pupil_opencv_sop05' not in globals():
    raise NameError("[ERROR] Fungsi deteksi belum tersedia. Jalankan SOP-04 (Langkah 1) terlebih dahulu!")

cap = cv2.VideoCapture(vid_lpw_path)
if not cap.isOpened():
    raise ValueError(f"Tidak dapat membuka video {vid_lpw_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or np.isnan(fps): fps = 95.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ret, first_frame = cap.read()
if not ret:
    raise ValueError("Video kosong.")
H_vid, W_vid, _ = first_frame.shape

# Sub-canvas dimensions (Grid 2x2)
PANEL_W, PANEL_H = W_vid, H_vid
GRID_W, GRID_H = PANEL_W * 2, PANEL_H * 2

# 1. PRA-KALKULASI BATAS Y-LIMITS TETAP (FIXED Y-LIMITS) DARI GROUND TRUTH
valid_gt_x = gt_df['GT_X'].dropna()
valid_gt_y = gt_df['GT_Y'].dropna()
valid_gt_x = valid_gt_x[valid_gt_x > 0]
valid_gt_y = valid_gt_y[valid_gt_y > 0]

if len(valid_gt_x) > 0:
    x_min_lim = max(0, float(valid_gt_x.min()) - 25)
    x_max_lim = min(float(W_vid), float(valid_gt_x.max()) + 25)
else:
    x_min_lim, x_max_lim = 0, float(W_vid)

if len(valid_gt_y) > 0:
    y_min_lim = max(0, float(valid_gt_y.min()) - 25)
    y_max_lim = min(float(H_vid), float(valid_gt_y.max()) + 25)
else:
    y_min_lim, y_max_lim = 0, float(H_vid)

print(f"[INFO] Batas Sumbu X Tetap : [{x_min_lim:.1f} px s/d {x_max_lim:.1f} px]")
print(f"[INFO] Batas Sumbu Y Tetap : [{y_min_lim:.1f} px s/d {y_max_lim:.1f} px]")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Super_Komparasi.mp4")
out_video = cv2.VideoWriter(out_vid_path, fourcc, fps, (GRID_W, GRID_H))

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

pred_x, pred_y, pred_d_list = [], [], []
euclidean_errors = []
last_valid = None
frames_lost = 0

limit_frames = min(total_frames, len(gt_df)) if len(gt_df) > 0 else total_frames

for i in tqdm(range(limit_frames), desc=f"Evaluasi LPW {lpw_id}"):
    ret, frame = cap.read()
    if not ret: break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    mask, diam, cx, cy, angle, last_valid, best_ellipse = detect_pupil_opencv_sop05(
        roi_gray=gray, px=None, py=None, last_valid_pupil=last_valid, frames_lost=frames_lost
    )
    
    if diam > 0:
        frames_lost = 0
    else:
        frames_lost += 1
        
    pred_x.append(cx)
    pred_y.append(cy)
    pred_d_list.append(diam)
    
    if i < len(gt_df):
        gx = gt_df['GT_X'].iloc[i]
        gy = gt_df['GT_Y'].iloc[i]
    else:
        gx, gy = np.nan, np.nan
        
    if cx > 0 and not np.isnan(gx) and not np.isnan(gy) and gx > 0:
        err = np.hypot(cx - gx, cy - gy)
    else:
        err = np.nan
    euclidean_errors.append(err)
    
    # --- 1. KIRI ATAS: ORIGINAL VIDEO + GROUND TRUTH MARKER ---
    tl_frame = frame.copy()
    if not np.isnan(gx) and not np.isnan(gy) and gx > 0:
        cv2.circle(tl_frame, (int(gx), int(gy)), 4, (0, 255, 0), -1, cv2.LINE_AA)
        cv2.drawMarker(tl_frame, (int(gx), int(gy)), (0, 0, 255), cv2.MARKER_CROSS, 14, 2)
        cv2.putText(tl_frame, f"1. GT LPW CENTER (X:{int(gx)}, Y:{int(gy)})", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
    else:
        cv2.putText(tl_frame, "1. GT LPW CENTER (BLINK / NO DATA)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)
        
    # --- 2. KIRI BAWAH: OPENCV TRACKING OVERLAY ---
    bl_frame = frame.copy()
    if cx > 0:
        if best_ellipse is not None and len(best_ellipse) == 3:
            cv2.ellipse(bl_frame, best_ellipse, (255, 0, 0), 2, cv2.LINE_AA)
        else:
            cv2.circle(bl_frame, (int(cx), int(cy)), int(diam/2), (255, 0, 0), 2, cv2.LINE_AA)
        cv2.circle(bl_frame, (int(cx), int(cy)), 3, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.putText(bl_frame, f"2. OPENCV TRACKING (D:{diam:.1f}px)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
        if not np.isnan(err):
            cv2.putText(bl_frame, f"Err:{err:.1f}px", (PANEL_W - 150, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2, cv2.LINE_AA)
    else:
        cv2.putText(bl_frame, "2. OPENCV TRACKING (BLINK / LOST)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)

    # --- 3. KANAN ATAS: REALTIME CHART SUMBU X (FIXED Y-LIMITS) ---
    fig_x, ax_x = plt.subplots(figsize=(PANEL_W/100, PANEL_H/100), dpi=100)
    x_indices = np.arange(i + 1)
    gt_x_vals = gt_df['GT_X'].iloc[:i+1].values if len(gt_df) >= i+1 else np.full(i+1, np.nan)
    pred_x_vals = np.array([x if x > 0 else np.nan for x in pred_x[:i+1]])
    
    ax_x.plot(x_indices, gt_x_vals, color='#38a169', linewidth=2, label='Ground Truth X (1.txt)')
    ax_x.plot(x_indices, pred_x_vals, color='#e53e3e', linestyle='--', linewidth=2, label='OpenCV Prediksi X')
    ax_x.set_xlim(0, limit_frames)
    ax_x.set_ylim(x_min_lim, x_max_lim)
    ax_x.set_title(f"3. Perbandingan Posisi Sumbu X - Frame {i:04d}", fontsize=11, fontweight='bold', pad=8)
    ax_x.set_xlabel("Frame ID", fontsize=9)
    ax_x.set_ylabel("Koordinat X (px)", fontsize=9)
    ax_x.grid(True, linestyle='--', alpha=0.5)
    ax_x.legend(loc='upper right', fontsize=8)
    fig_x.tight_layout()
    
    fig_x.canvas.draw()
    tr_img = np.asarray(fig_x.canvas.buffer_rgba())[:, :, :3]
    tr_img = cv2.cvtColor(tr_img, cv2.COLOR_RGB2BGR)
    plt.close(fig_x)
    if tr_img.shape[0] != PANEL_H or tr_img.shape[1] != PANEL_W:
        tr_img = cv2.resize(tr_img, (PANEL_W, PANEL_H))

    # --- 4. KANAN BAWAH: REALTIME CHART SUMBU Y (FIXED Y-LIMITS) ---
    fig_y, ax_y = plt.subplots(figsize=(PANEL_W/100, PANEL_H/100), dpi=100)
    gt_y_vals = gt_df['GT_Y'].iloc[:i+1].values if len(gt_df) >= i+1 else np.full(i+1, np.nan)
    pred_y_vals = np.array([y if y > 0 else np.nan for y in pred_y[:i+1]])
    
    ax_y.plot(x_indices, gt_y_vals, color='#38a169', linewidth=2, label='Ground Truth Y (1.txt)')
    ax_y.plot(x_indices, pred_y_vals, color='#3182ce', linestyle='--', linewidth=2, label='OpenCV Prediksi Y')
    ax_y.set_xlim(0, limit_frames)
    ax_y.set_ylim(y_min_lim, y_max_lim)
    ax_y.set_title(f"4. Perbandingan Posisi Sumbu Y - Frame {i:04d}", fontsize=11, fontweight='bold', pad=8)
    ax_y.set_xlabel("Frame ID", fontsize=9)
    ax_y.set_ylabel("Koordinat Y (px)", fontsize=9)
    ax_y.grid(True, linestyle='--', alpha=0.5)
    ax_y.legend(loc='upper right', fontsize=8)
    fig_y.tight_layout()
    
    fig_y.canvas.draw()
    br_img = np.asarray(fig_y.canvas.buffer_rgba())[:, :, :3]
    br_img = cv2.cvtColor(br_img, cv2.COLOR_RGB2BGR)
    plt.close(fig_y)
    if br_img.shape[0] != PANEL_H or br_img.shape[1] != PANEL_W:
        br_img = cv2.resize(br_img, (PANEL_W, PANEL_H))

    top_row = np.hstack([tl_frame, tr_img])
    bot_row = np.hstack([bl_frame, br_img])
    grid_canvas = np.vstack([top_row, bot_row])
    
    out_video.write(grid_canvas)

cap.release()
out_video.release()
print(f"\n[INFO] Render Video Grid 4-Panel (2x2) Selesai! Tersimpan di: {out_vid_path}")


In [ ]:
# @title 📊 SOP-07 (Langkah 4): Ekspor pred.txt, 3 Grafik PNG, CSV & PDF Report
# ======================================================================
# SOP-07 (Langkah 4): EKSPOR PRED.TXT, 3 GRAFIK PNG, CSV & REPORTLAB PDF
# ======================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"\n======================================================================")
print(f"[TARGET LPW: {lpw_id}] - EKSPOR PRED.TXT, 3 GRAFIK PNG, CSV & PDF REPORT")
print(f"======================================================================")

# 1. Ekspor Berkas pred.txt Murni 2-Kolom (Format standar LPW)
pred_txt_path = os.path.join(out_dir_lpw, "pred.txt")
with open(pred_txt_path, "w", encoding="utf-8") as f_pred:
    for x_val, y_val in zip(pred_x, pred_y):
        f_pred.write(f"{float(x_val):.2f} {float(y_val):.2f}\n")
print(f"✅ [1/5] Berkas pred.txt murni 2-kolom tersimpan di: {pred_txt_path}")

# 2. Simpan Spreadsheet CSV Metrics
df_metrics = pd.DataFrame({
    'Frame': range(1, len(euclidean_errors) + 1),
    'GT_X': gt_df['GT_X'].iloc[:len(euclidean_errors)].values if len(gt_df) >= len(euclidean_errors) else np.nan,
    'GT_Y': gt_df['GT_Y'].iloc[:len(euclidean_errors)].values if len(gt_df) >= len(euclidean_errors) else np.nan,
    'Pred_X': pred_x,
    'Pred_Y': pred_y,
    'Pred_D': pred_d_list,
    'Center_Error_px': euclidean_errors,
    'Is_Blink': [1 if d == 0 else 0 for d in pred_d_list]
})

csv_out_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Benchmark_Report.csv")
df_metrics.to_csv(csv_out_path, index=False)
print(f"✅ [2/5] Berkas Spreadsheet CSV tersimpan di: {csv_out_path}")

# 3. Hitung Metrik Kuantitatif
valid_errs = df_metrics['Center_Error_px'].dropna()
mean_err = float(valid_errs.mean()) if len(valid_errs) > 0 else 0.0
median_err = float(valid_errs.median()) if len(valid_errs) > 0 else 0.0
detection_rate_5px = float((valid_errs < 5.0).mean() * 100.0) if len(valid_errs) > 0 else 0.0
detection_rate_10px = float((valid_errs < 10.0).mean() * 100.0) if len(valid_errs) > 0 else 0.0

print(f"\n=== HASIL EVALUASI BENCHMARK LPW [{lpw_id}] ===")
print(f"- Mean Center Error (px)  : {mean_err:.2f} px")
print(f"- Median Center Error (px) : {median_err:.2f} px")
print(f"- Detection Rate (< 5px)  : {detection_rate_5px:.2f} %")
print(f"- Detection Rate (< 10px) : {detection_rate_10px:.2f} %")

# 4. Render 3 Berkas Grafik PNG Statis
# --- Grafik 1: Perbandingan Sumbu X ---
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(df_metrics['Frame'], df_metrics['GT_X'], color='#38a169', linewidth=1.5, label='Ground Truth X (1.txt)')
plt.plot(df_metrics['Frame'], df_metrics['Pred_X'].replace(0, np.nan), color='#e53e3e', linestyle='--', linewidth=1.5, label='OpenCV Prediksi X')
plt.title(f'Perbandingan Sinyal Trajektori Sumbu X - LPW {lpw_id}', fontsize=10, fontweight='bold')
plt.xlabel('Frame ID', fontsize=8)
plt.ylabel('Koordinat X (px)', fontsize=8)
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
chart_x_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Sumbu_X_Chart.png")
plt.savefig(chart_x_path, dpi=150)
plt.show()

# --- Grafik 2: Perbandingan Sumbu Y ---
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(df_metrics['Frame'], df_metrics['GT_Y'], color='#38a169', linewidth=1.5, label='Ground Truth Y (1.txt)')
plt.plot(df_metrics['Frame'], df_metrics['Pred_Y'].replace(0, np.nan), color='#3182ce', linestyle='--', linewidth=1.5, label='OpenCV Prediksi Y')
plt.title(f'Perbandingan Sinyal Trajektori Sumbu Y - LPW {lpw_id}', fontsize=10, fontweight='bold')
plt.xlabel('Frame ID', fontsize=8)
plt.ylabel('Koordinat Y (px)', fontsize=8)
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
chart_y_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Sumbu_Y_Chart.png")
plt.savefig(chart_y_path, dpi=150)
plt.show()

# --- Grafik 3: Ground Truth Distance Error ---
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(df_metrics['Frame'], df_metrics['Center_Error_px'], label='Center Distance Error (px)', color='#d53f8c', linewidth=1.2)
plt.axhline(5.0, color='#38a169', linestyle='--', linewidth=1.5, label='Threshold Presisi 5px')
plt.title(f'Ground Truth Benchmark Validation LPW {lpw_id}\nMean Error: {mean_err:.2f} px | Detection Rate (<5px): {detection_rate_5px:.2f} %', fontsize=10, fontweight='bold')
plt.xlabel('Frame ID', fontsize=8)
plt.ylabel('Center Distance Error (px)', fontsize=8)
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
chart_error_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Error_Chart.png")
plt.savefig(chart_error_path, dpi=150)
plt.show()
print(f"✅ [3/5] Berhasil menyimpan 3 grafik PNG statis (Sumbu X, Sumbu Y, & Error Chart)!")

# 5. Menerbitkan Dokumen PDF Evaluasi Resmi (ReportLab)
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'reportlab', '--quiet'])
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors

pdf_out_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Laporan_Evaluasi_GT.pdf")
doc = SimpleDocTemplate(pdf_out_path, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
styles = getSampleStyleSheet()

title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=16, leading=20, textColor=colors.HexColor('#003366'))
sub_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=9.5, leading=13, textColor=colors.HexColor('#444444'))
body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontName='Helvetica', fontSize=9, leading=12, textColor=colors.HexColor('#222222'))

story = []
story.append(Paragraph(f"LAPORAN EVALUASI BENCHMARK LPW - DATASET {lpw_id}", title_style))
story.append(Paragraph(f"<b>Algoritma:</b> OpenCV Pupil Detector | <b>Dataset:</b> Labeled Pupils in the Wild (LPW {lpw_id})", sub_style))
story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#003366"), spaceAfter=8))

tbl_data = [
    [Paragraph("<b>Metrik Evaluasi</b>", body_style), Paragraph("<b>Nilai Hasil</b>", body_style)],
    [Paragraph("Mean Center Error (Rata-rata)", body_style), Paragraph(f"{mean_err:.2f} Piksel", body_style)],
    [Paragraph("Median Center Error (Nilai Tengah)", body_style), Paragraph(f"{median_err:.2f} Piksel", body_style)],
    [Paragraph("Detection Rate (Error < 5px)", body_style), Paragraph(f"{detection_rate_5px:.2f} %", body_style)],
    [Paragraph("Detection Rate (Error < 10px)", body_style), Paragraph(f"{detection_rate_10px:.2f} %", body_style)],
    [Paragraph("Total Frame Valid", body_style), Paragraph(f"{len(valid_errs)} Frame", body_style)]
]

t = Table(tbl_data, colWidths=[280, 260])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#EAECEE")),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#BDC3C7")),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE')
]))
story.append(t)
story.append(Spacer(1, 10))

if os.path.exists(chart_error_path):
    story.append(Paragraph("Grafik Trajektori Center Distance Error (px):", styles['Heading2']))
    story.append(RLImage(chart_error_path, width=540, height=225))

doc.build(story)

print(f"======================================================================")
print(f"✅ EVALUASI BENCHMARK LPW SELESAI!")
print(f"📁 Folder Output  : Output_LPW_{lpw_id}")
print(f"📄 File pred.txt   : pred.txt")
print(f"📄 File PDF Report: {os.path.basename(pdf_out_path)}")
print(f"======================================================================\n")


### <font color="#00c2cb" face="Palatino Linotype">**SOP-08: Penyimpanan & Tata Kelola Dataset (Private Research Storage)**</font>

**Deskripsi:**
Tata kelola penyimpanan dataset internal tim riset (Google Drive & Harddisk 1TB Privat) yang berstatus **Restricted / Private**.

### <font color="#00c2cb" face="Palatino Linotype">**SOP-09: Dokumentasi Riset & Publikasi Jurnal Scientific**</font>

**Deskripsi:**
Penyusunan naskah akademik manuskrip jurnal ilmiah dan proposal hibah penelitian berdasarkan luaran analisis otomasisasi 9-SOP.
